RECOMMENDATION SYSTEM

1. RECOMMENDATIONS BUILD

1.1 Basket Creation : Strength Score

In [ ]:
# Recent purchases = higher weight
df['Recency'] = (
    df['BILL_DATE'].max()
    - df['BILL_DATE']
).dt.days

df['recency_weight'] = (
    1 / (df['Recency'] + 1)
)

In [ ]:
# Value weight

df['value_weight'] = (
    df['NSV']
    / (df['QTY'] + 1)
)

In [ ]:
# Normalizing value weight
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df['value_weight'] = scaler.fit_transform(
    df[['value_weight']]
)

In [ ]:
# Customer-Category Frequency

customer_product_freq = df.groupby(
    ['CUSTOMER_ID', 'PRODUCT_DESCRIPTION']
).size().reset_index(name='purchase_frequency')

# Merge back
df = df.merge(
    customer_product_freq,
    on=['CUSTOMER_ID', 'PRODUCT_DESCRIPTION'],
    how='left'
)

In [ ]:
# Normalizing frequency weight

df['frequency_weight'] = (
    df['purchase_frequency']
    /
    df['purchase_frequency'].max()
)


In [ ]:
# FINAL INTERACTION WEIGHT : RFM Based

df['final_weight'] = (
    0.4 * df['recency_weight']
    +
    0.3 * df['value_weight']
    +
    0.3 * df['frequency_weight']
)

In [ ]:
df[['recency_weight',
    'value_weight',
    'frequency_weight',
    'final_weight','QTY']].describe()

,recency_weight,value_weight,frequency_weight,final_weight,QTY
count,23865.000000,23865.000000,23865.000000,23865.000000,23865.000000
mean,0.124242,0.087304,0.201108,0.136220,1.797528
std,0.123831,0.058094,0.360292,0.117860,1.404247
min,0.041667,0.000000,0.000281,0.018351,1.000000
25%,0.055556,0.058942,0.000281,0.052427,1.000000
50%,0.076923,0.076547,0.000281,0.078534,1.000000
75%,0.142857,0.102536,0.279011,0.186441,2.000000
max,1.000000,1.000000,1.000000,0.744304,69.000000


In [ ]:
# Basket with Final weight
basket = pd.pivot_table(
    df,
    index='CUSTOMER_ID',
    columns='PRODUCT_DESCRIPTION',
    values='final_weight',
    aggfunc='sum'
).fillna(0)

In [ ]:
df.columns

Index(['BILL_NO', 'CUSTOMER_ID', 'PRODUCT_DESCRIPTION', 'BILL_DATE', 'NSV',
       'QTY', 'DISCOUNT', 'Previous Date', 'Gap Days', 'Recency',
       'recency_weight', 'value_weight', 'purchase_frequency',
       'frequency_weight', 'final_weight'],
      dtype='object')

1.2 Similarity

In [ ]:
# Normalize customer-product matrix
from sklearn.preprocessing import normalize

basket_normalized = normalize(
    basket,
    norm='l2'
)

basket_normalized = pd.DataFrame(
    basket_normalized,
    index=basket.index,
    columns=basket.columns
)

In [ ]:
# Product similarity matrix
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(
    basket_normalized.T
)

sim_df = pd.DataFrame(
    similarity,
    index=basket_normalized.columns,
    columns=basket_normalized.columns
)

In [ ]:
sim_df.head()

PRODUCT_DESCRIPTION,Accessories,Jacket,Jeans,Shirt,Suit,T-Shirt,Trouser
PRODUCT_DESCRIPTION,,,,,,,
Accessories,1.000000,0.010779,0.001357,0.015434,0.055409,0.002760,0.008024
Jacket,0.010779,1.000000,0.007359,0.030631,0.010609,0.008298,0.027701
Jeans,0.001357,0.007359,1.000000,0.139673,0.001116,0.082588,0.085850
Shirt,0.015434,0.030631,0.139673,1.000000,0.036936,0.196979,0.299834
Suit,0.055409,0.010609,0.001116,0.036936,1.000000,0.002672,0.007256


1.3 Recommendation Function

In [ ]:
# TOP PRODUCTS (fallback recommendations)

top_products = (
    df['PRODUCT_DESCRIPTION']
    .value_counts()
    .index
    .tolist()
)

# RECOMMENDATION FUNCTION

def recommend_for_customer(
    customer_id,
    upsell_flag,
    top_n=3
):

    # Products already purchased
    customer_products = df[
        df['CUSTOMER_ID'] == customer_id
    ]['PRODUCT_DESCRIPTION'].unique()

    # Dictionary for recommendation scores
    recs = {}


    # ITEM-ITEM RECOMMENDATIONS
    for product in customer_products:


        if product not in sim_df.columns:
            continue

        # Get similar products
        similar_products = (
            sim_df[product]
            .sort_values(ascending=False)
            .iloc[1:6]
        )

        # Add weighted similarity score
        for item, score in similar_products.items():

            # Avoids recommending already purchased items
            if item not in customer_products:

                recs[item] = (
                    recs.get(item, 0)
                    + score
                )


    # SORT RECOMMENDATIONS

    recs_sorted = sorted(
        recs.items(),
        key=lambda x: x[1],
        reverse=True
    )

    recommended_products = [
        item[0]
        for item in recs_sorted
    ]


    # BUSINESS LOGIC LAYER

    # High upsell customers
    if upsell_flag == 1:

        # Stronger recommendations
        return (
            recommended_products[:top_n]
            if recommended_products
            else top_products[:top_n]
        )

    # Lower upsell customers
    else:

        # Softer / safer recommendations
        return (
            recommended_products[:2]
            if recommended_products
            else top_products[:2]
        )


In [ ]:
recommend_for_customer(
    customer_id='some_id',
    upsell_flag=1
)

['Shirt', 'Trouser', 'T-Shirt']

In [ ]:
# PRODUCTS PURCHASED BY CUSTOMER

customer_products = df.groupby(
    "CUSTOMER_ID"
)["PRODUCT_DESCRIPTION"].apply(
    lambda x: list(set(x))
)

# Map into features table
features["Purchased_Products"] = (
    features["CUSTOMER_ID"]
    .map(customer_products)
)